In [0]:
%pip install statsmodels

In [0]:
# =============================================================================
# NB09 — Feature selection (fast: correlation filter + MDI importance)
# =============================================================================
# Objective:
#   Identify the most informative features for each layer (L1, L2, L3).
#
#   Two-step selection:
#     1. Correlation filter — drop features with |r| > 0.90 with any higher-
#        ranked feature (rank = MDI importance). Faster equivalent of VIF for
#        this dataset size. Applied on the full training set (fold 0).
#
#     2. MDI importance — RF feature_importances_ on fold-0 training data.
#        Top-N features selected per layer (N=40 for L1/L2, N=50 for L3).
#        Physical must-keeps (structural, Butler-Ginley, key experimentals)
#        are always included regardless of rank.
#
#   Both full and selected feature sets are written so NB10 can compare them.
#
# Inputs:
#   ml_perovskites.master, cv_splits, X_material, X_thermo, X_exp
#
# Outputs:
#   ml_perovskites.feature_importance   (layer, feature, mdi_importance)
#   ml_perovskites.selected_features    (layer, feature)
#   /tmp/nb09_importance_L1.png
#   /tmp/nb09_importance_L3.png
#
# Rules: no leakage — all selection computed on training fold only. SEED=42.
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import copy

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute   import SimpleImputer
from pyspark.sql import SparkSession

SEED       = 42
SCHEMA     = "ml_perovskites"
CORR_THRESH = 0.90   # drop feature if |r| > this with a higher-ranked feature
TOP_N_L1L2 = 40
TOP_N_L3   = 50
CAT_COLS   = ["co_catalyst_type", "run_type", "scavenger_class", "lamp_type"]

# Physical must-keep features (always included regardless of importance rank)
MUST_KEEP_PATTERNS = [
    "goldschmidt", "octahedral", "Ra_Rb_ratio",
    "ECB_butler", "EVB_butler", "chi_mulliken",
    "co_catalyst", "scavenger", "run_type", "lamp_type",
    "catalyst_amount",
]

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

# ---------------------------------------------------------------------------
# 1. Load tables
# ---------------------------------------------------------------------------
master    = spark.table(f"{SCHEMA}.master").toPandas()
cv_splits = spark.table(f"{SCHEMA}.cv_splits").toPandas()
X_mat_raw = spark.table(f"{SCHEMA}.X_material").toPandas()
X_thm_raw = spark.table(f"{SCHEMA}.X_thermo").toPandas()
X_exp_raw = spark.table(f"{SCHEMA}.X_exp").toPandas()

def is_use_h2(series):
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["true", "1"])

master["y_log10_1p_final"] = pd.to_numeric(master["y_log10_1p_final"], errors="coerce")

df_base = master[
    is_use_h2(master["use_h2"]) &
    master["y_log10_1p_final"].notna() &
    master["doi_key"].notna()
].copy()
df_base["y"] = df_base["y_log10_1p_final"].astype(float)
print(f"Modelling rows: {len(df_base)}")

META_COLS = {"source_row_id", "base_compound", "family_proposed",
             "formula_base_paper", "A_site_paper", "B_site_paper",
             "doi", "doi_key", "y_log10_1p_final", "y_log10_1p",
             "h2_rate_umol_g_h", "h2_umol_h_rate_mass"}

def numeric_features(df):
    return [c for c in df.columns
            if c not in META_COLS and pd.api.types.is_numeric_dtype(df[c])]

# Join key must have the same dtype in every table — force string everywhere
for t in [df_base, cv_splits, X_mat_raw, X_thm_raw, X_exp_raw]:
    t["source_row_id"] = t["source_row_id"].astype(str).str.strip()

# Convert Decimal→float for feature columns only (never the join key)
for raw in [X_mat_raw, X_thm_raw]:
    for c in raw.columns:
        if c == "source_row_id":
            continue
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

x_mat_cols = numeric_features(X_mat_raw)
X_mat = X_mat_raw[["source_row_id"] + x_mat_cols].copy()
high_nan   = [c for c in x_mat_cols if X_mat[c].isna().mean() >= 0.50]
X_mat.drop(columns=high_nan, inplace=True)
x_mat_cols = [c for c in x_mat_cols if c not in high_nan]

x_thm_cols = numeric_features(X_thm_raw)
X_thm = X_thm_raw[["source_row_id"] + x_thm_cols].copy()
zero_fill  = [c for c in x_thm_cols if X_thm[c].isna().mean() >= 0.99]
X_thm.drop(columns=zero_fill, inplace=True)
x_thm_cols = [c for c in x_thm_cols if c not in zero_fill]

x_exp_num = [c for c in X_exp_raw.columns
             if c not in ["source_row_id"] + CAT_COLS
             and pd.api.types.is_numeric_dtype(X_exp_raw[c])]
X_exp = X_exp_raw[["source_row_id"] + x_exp_num + CAT_COLS].copy()
for c in x_exp_num:
    X_exp[c] = pd.to_numeric(X_exp[c], errors="coerce")

df = (df_base[["source_row_id", "doi_key", "y"]]
      .merge(X_mat, on="source_row_id", how="left")
      .merge(X_thm, on="source_row_id", how="left")
      .merge(X_exp, on="source_row_id", how="left"))
print(f"Merged shape: {df.shape}")

L1_COLS = x_mat_cols
L2_COLS = x_mat_cols + x_thm_cols
L3_NUM  = x_mat_cols + x_thm_cols + x_exp_num

main_cv = cv_splits[cv_splits["split_type"] == "main_cv"]
fold0   = main_cv[main_cv["fold_id"] == 0]
tr_ids  = set(fold0[fold0["role"] == "train"]["source_row_id"])
tr      = df[df["source_row_id"].isin(tr_ids)].copy()
print(f"Fold-0 training rows: {len(tr)}")

# ---------------------------------------------------------------------------
# 2. Smoothed target encoding (for L3 categoricals, on fold-0 train)
# ---------------------------------------------------------------------------
def target_encode_train(tr_df, cat_col, m=10):
    gm = tr_df["y"].mean()
    stats = tr_df.groupby(cat_col)["y"].agg(["mean", "count"]).reset_index()
    stats["enc"] = (stats["mean"] * stats["count"] + gm * m) / (stats["count"] + m)
    return tr_df[cat_col].map(stats.set_index(cat_col)["enc"]).fillna(gm)

# ---------------------------------------------------------------------------
# 3. MDI importance (RF on fold-0 train, per layer)
# ---------------------------------------------------------------------------
print("\n--- MDI feature importance (RF 300 trees, fold-0 train) ---")
imp_records = []

rf = RandomForestRegressor(n_estimators=300, max_features="sqrt",
                           min_samples_leaf=3, oob_score=True,
                           n_jobs=-1, random_state=SEED)

for layer_name, feat_cols in [("L1", L1_COLS), ("L2", L2_COLS), ("L3", L3_NUM)]:
    X_tr = tr[feat_cols].copy()

    if layer_name == "L3":
        for cc in CAT_COLS:
            X_tr[cc] = target_encode_train(tr, cc)

    imp_fill = SimpleImputer(strategy="median")
    X_arr    = imp_fill.fit_transform(X_tr.values.astype(float))

    m = copy.deepcopy(rf)
    m.fit(X_arr, tr["y"].values)

    oob = m.oob_score_
    print(f"  {layer_name}: {len(feat_cols)} features | OOB R²={oob:.4f}")

    for feat, imp_val in zip(feat_cols, m.feature_importances_):
        imp_records.append({
            "layer":           layer_name,
            "feature":         feat,
            "mdi_importance":  float(imp_val),
        })

imp_df = pd.DataFrame(imp_records)

# ---------------------------------------------------------------------------
# 4. Correlation filter (on fold-0 train numeric features)
# ---------------------------------------------------------------------------
print("\n--- Correlation filter (|r| > 0.90, fold-0 train) ---")
corr_selected = {}

for layer_name, feat_cols in [("L1", L1_COLS), ("L2", L2_COLS), ("L3", L3_NUM)]:
    # Sort by MDI descending — keep higher-ranked feature when pair is correlated
    ranked = (imp_df[imp_df["layer"] == layer_name]
              .sort_values("mdi_importance", ascending=False)["feature"]
              .tolist())

    X_tr_df = tr[ranked].copy()
    # For L3, replace categoricals with encoded versions
    if layer_name == "L3":
        for cc in CAT_COLS:
            if cc in X_tr_df.columns:
                X_tr_df[cc] = target_encode_train(tr, cc)

    imp_fill = SimpleImputer(strategy="median")
    X_imp    = pd.DataFrame(
        imp_fill.fit_transform(X_tr_df.values.astype(float)),
        columns=ranked)

    corr_mat = X_imp.corr().abs()
    keep = []
    for feat in ranked:
        if feat not in keep:
            # Check against already-kept features
            if keep:
                max_corr = corr_mat.loc[feat, keep].max()
                if max_corr > CORR_THRESH:
                    continue   # skip — too correlated with a better feature
            keep.append(feat)

    removed = [c for c in feat_cols if c not in keep]
    print(f"  {layer_name}: {len(feat_cols)} → {len(keep)} features "
          f"({len(removed)} removed by |r|>{CORR_THRESH})")
    corr_selected[layer_name] = keep

# ---------------------------------------------------------------------------
# 5. Build selected feature sets (corr-filter ∩ top-N, plus must-keeps)
# ---------------------------------------------------------------------------
print("\n--- Selected feature sets ---")
sel_records = []

for layer_name, feat_cols in [("L1", L1_COLS), ("L2", L2_COLS), ("L3", L3_NUM)]:
    corr_set  = set(corr_selected.get(layer_name, feat_cols))
    top_n     = TOP_N_L3 if layer_name == "L3" else TOP_N_L1L2

    ranked = (imp_df[imp_df["layer"] == layer_name]
              .sort_values("mdi_importance", ascending=False)["feature"]
              .tolist())
    top_set = set(ranked[:top_n])

    # Intersection of corr-survivors and top-N
    selected = [c for c in ranked if c in corr_set and c in top_set]

    # Force-include physical must-keeps from this layer
    must = [c for c in feat_cols
            if any(k in c for k in MUST_KEEP_PATTERNS) and c not in selected]
    selected = list(dict.fromkeys(selected + must))

    print(f"  {layer_name}: {len(selected)} features selected "
          f"(from {len(feat_cols)} total)")

    for feat in selected:
        sel_records.append({"layer": layer_name, "feature": feat})

selected_df = pd.DataFrame(sel_records)

# ---------------------------------------------------------------------------
# 6. Plots (top-30 MDI per layer)
# ---------------------------------------------------------------------------
for layer_name in ["L1", "L3"]:
    sub = (imp_df[imp_df["layer"] == layer_name]
           .sort_values("mdi_importance", ascending=False).head(30))
    fig, ax = plt.subplots(figsize=(8, 10))
    colors  = ["#2196F3" if f in set(corr_selected.get(layer_name, []))
               else "#BDBDBD" for f in sub["feature"][::-1]]
    ax.barh(sub["feature"][::-1], sub["mdi_importance"][::-1], color=colors)
    ax.set_xlabel("MDI importance")
    ax.set_title(f"Top 30 features — {layer_name}  "
                 f"(blue = corr-selected, grey = removed)")
    plt.tight_layout()
    path = f"/tmp/nb09_importance_{layer_name}.png"
    plt.savefig(path, dpi=120)
    plt.close()
    print(f"Saved: {path}")

# ---------------------------------------------------------------------------
# 7. Write Delta tables
# ---------------------------------------------------------------------------
spark.createDataFrame(imp_df.replace({np.nan: None})).write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA}.feature_importance")

spark.createDataFrame(selected_df).write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA}.selected_features")

n_imp = spark.table(f"{SCHEMA}.feature_importance").count()
n_sel = spark.table(f"{SCHEMA}.selected_features").count()
print(f"\n✓ Written: {SCHEMA}.feature_importance ({n_imp} rows)")
print(f"✓ Written: {SCHEMA}.selected_features ({n_sel} rows)")
print("\n=== NB09 COMPLETE ===")
print("Next: NB10 — HPO and full CV stack using selected features.")